# Week 3 — RAG (Retrieval-Augmented Generation), step by step

In the HR assistant we pasted the **entire** IT policy into the system prompt on every call.
That works for one small file, but it breaks down quickly:

| Problem with "stuff the whole doc" | What RAG does instead |
|---|---|
| Every call pays for ~5.5k tokens of policy, even for "hi" | Send only the 3–5 most relevant paragraphs |
| Add 50 more policies → blows past the context window | Index any number of documents, retrieve a few |
| Model gets "lost in the middle" of long context | Small, focused context = better answers |
| Hard to tell *where* an answer came from | Each chunk carries source + section → citations |

**The RAG pipeline**

```mermaid
flowchart LR
    subgraph IDX["INDEXING (offline, once)"]
        direction TB
        D["docs/*.txt"] -->|"1. chunk"| C["Chunks<br/>(text + metadata)"]
        C -->|"2. embed"| E["Vectors<br/>(1536 floats each)"]
    end

    E -->|"3. store"| DB[("Chroma DB<br/>./chroma_db/")]

    subgraph QRY["QUERYING (online, every question)"]
        direction TB
        Q["User question"] -->|"4. embed"| QV["Question vector"]
        QV -->|"5. top-K search"| DB
        DB -->|"K most similar chunks"| P["Prompt =<br/>instructions + context + question"]
        P -->|"6. call LLM"| A["Grounded answer<br/>with citations"]
    end
```

We'll build each step by hand using the policy docs in `hr-assistant/docs/`.

## Step 0 — Setup

In [1]:
# pip install chromadb openai python-dotenv tiktoken
import os, re
from pathlib import Path

import chromadb
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from .env
client = OpenAI()

DOCS_DIR = Path("../hr-assistant/docs")
CHROMA_DIR = Path("./chroma_db")       # Chroma will persist here (just a folder on disk)

EMBED_MODEL = "text-embedding-3-small"  # 1536-dim, cheap ($0.02 / 1M tokens)
CHAT_MODEL = "gpt-4o-mini"

enc = tiktoken.get_encoding("cl100k_base")  # tokenizer used by the embedding models
def n_tokens(text: str) -> int:
    return len(enc.encode(text))

### Load the documents
`docs/` has `it-assets-policy.txt` (and an empty `leave-policy.txt` — we skip empty files).

In [2]:
documents = {}
for path in sorted(DOCS_DIR.glob("*.txt")):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        print(f"skipping empty file: {path.name}")
        continue
    documents[path.name] = text
    print(f"{path.name}: {len(text):,} chars, {n_tokens(text):,} tokens")

it-assets-policy.txt: 28,513 chars, 5,456 tokens


In [3]:
print(documents["it-assets-policy.txt"][:900])

IT ASSETS POLICY

Document Owner: IT Department (in coordination with Human Resources)
Version: 1.0
Effective Date: 1 October 2026
Review Cycle: Annually
Applies To: All employees, interns, contractors and consultants of the Company

1. PURPOSE

This policy explains how the Company issues, manages, protects and recovers
its IT assets. It sets out what employees can expect to receive, how they
must look after Company equipment, and what happens when equipment is lost,
damaged, upgraded or returned.

The goals of this policy are to:
- Give every employee the tools they need to do their job well.
- Protect Company and customer data stored on or accessed through devices.
- Keep a clear record of who holds which asset at all times.
- Control costs by ex


---
## Step 1 — Chunking

An embedding is **one vector per piece of text**. If we embed the whole ~5.5k-token policy as one vector,
its meaning becomes a blurry average of 20 topics and search can't pinpoint anything.
So we split the document into **chunks** — small, self-contained pieces about one thing.

Trade-off:
- **Too big** → noisy vector, wastes prompt tokens, retrieves irrelevant text along with the relevant bit.
- **Too small** → loses context ("within 24 hours" — of what?).
- Typical sweet spot: **200–500 tokens**, with a little **overlap** so sentences cut at a boundary aren't lost.

We'll look at two strategies.

### 1a. Naive: fixed-size chunks with overlap
Simplest possible approach — slide a window of N tokens with some overlap. Doesn't care about structure at all.

In [5]:
def fixed_size_chunks(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    tokens = enc.encode(text)
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(tokens), step):
        window = tokens[start : start + chunk_size]
        chunks.append(enc.decode(window))
        if start + chunk_size >= len(tokens):
            break
    return chunks

naive = fixed_size_chunks(documents["it-assets-policy.txt"])
print(f"{len(naive)} chunks\n")
print("── chunk 3 ──────────────────────────")
print(naive[3])

22 chunks

── chunk 3 ──────────────────────────
 lost or damaged assets where applicable.

4.5 Finance
- Approve budgets for asset purchases.
- Process cost recoveries and allowance reimbursements.
- Maintain depreciation records.

5. ASSET ENTITLEMENT

5.1 Standard Entitlement by Role

The table below shows the standard equipment issued. Actual specifications
may change based on availability and current market models.

Role Category                 | Laptop                 | Mobile Phone       | Other
------------------------------|------------------------|--------------------|---------------------------
General / Operations staff    | Standard laptop        | Not provided       | Headset, mouse
Software Engineers / Data     | High-performance       | Not provided       | 1 external monitor,
                              | laptop (min. 16 GB RAM,|                    | keyboard, mouse, headset
                              | 512 GB SSD)            |                    |
Designers / Cr

Notice the problem: the chunk starts and ends **mid-sentence / mid-section**, and may mix the tail of one
section with the start of another. The retriever can still work, but the chunks are messy.

### 1b. Structure-aware: split on the document's own sections
Our policy has a clear structure:
```
======================================
6. LAPTOPS AND DESKTOPS          <- section
======================================
6.1 Standard Configuration       <- sub-section
...
```
Better strategy:
1. Split into **sections** on the `=====` headers.
2. Split each section into **sub-sections** (`6.1`, `6.2`, ...).
3. If a piece is still larger than `max_tokens`, fall back to splitting on paragraphs.
4. **Prefix every chunk with its section title** so a chunk like "Report within 24 hours" still carries
   "12. LOSS, THEFT AND DAMAGE" — this context gets baked into the embedding.
5. Keep **metadata** (source file, section, chunk index) — we'll use it for citations and filtering.

In [ ]:
SECTION_RE = re.compile(r"^=+\n(\d+\.\s+[^\n]+)\n=+\n", re.MULTILINE)   # === / "6. TITLE" / ===
SUBSECTION_RE = re.compile(r"^(?=\d+\.\d+\s)", re.MULTILINE)             # line starting with "6.1 "

def split_sections(text: str) -> list[tuple[str, str]]:
    # re.split with a capture group returns: [preamble, title1, body1, title2, body2, ...]
    parts = SECTION_RE.split(text)
    sections = [("PREAMBLE", parts[0].strip())] if parts[0].strip() else []
    for title, body in zip(parts[1::2], parts[2::2]):
        sections.append((title.strip(), body.strip()))
    return sections

def chunk_document(source: str, text: str, max_tokens: int = 350) -> list[dict]:
    chunks = []
    for section_title, body in split_sections(text):
        subsections = [s.strip() for s in SUBSECTION_RE.split(body) if s.strip()]
        for sub in subsections:
            chunks.append({
                "id": f"{source}::{len(chunks)}",
                "text": f"[{section_title}]\n{sub}",   # section title prefixed for context
                "metadata": {"source": source, "section": section_title, "chunk_index": len(chunks)},
            })
    return chunks

all_chunks = []
for source, text in documents.items():
    all_chunks.extend(chunk_document(source, text))

sizes = [n_tokens(c["text"]) for c in all_chunks]
print(f"{len(all_chunks)} chunks | min {min(sizes)} / avg {sum(sizes)//len(sizes)} / max {max(sizes)} tokens")

57 chunks | min 38 / avg 100 / max 361 tokens


In [6]:
for c in all_chunks[10:13]:
    print(f"── {c['id']}  {c['metadata']}")
    print(c["text"], "\n")

── it-assets-policy.txt::10  {'source': 'it-assets-policy.txt', 'section': '5. ASSET ENTITLEMENT', 'chunk_index': 10}
[5. ASSET ENTITLEMENT]
5.2 Additional Equipment
Employees may request additional equipment (for example, a second monitor or
an ergonomic chair accessory) through the IT ticketing portal. Requests need
manager approval and, for items above the departmental limit, Finance
approval. Requests on medical grounds should be supported by a doctor's note
and are prioritised. 

── it-assets-policy.txt::11  {'source': 'it-assets-policy.txt', 'section': '5. ASSET ENTITLEMENT', 'chunk_index': 11}
[5. ASSET ENTITLEMENT]
5.3 Only One Primary Device
Each employee is normally issued one primary laptop. A second laptop is given
only where there is a documented business need (for example, testing on a
different operating system) and requires department head approval. 

── it-assets-policy.txt::12  {'source': 'it-assets-policy.txt', 'section': '6. LAPTOPS AND DESKTOPS', 'chunk_index': 12}

Each chunk is now one clean idea (e.g. "6.1 Standard Configuration") tagged with its section.
That is what we'll embed.

### 1c. (FYI) LLM-assisted chunking — not needed for our doc

When a document has **no reliable structure** (meeting transcripts, scanned PDFs, emails, long prose with no headings),
regex rules like 1b can't find good boundaries. You can instead ask an **LLM to do the chunking**: give it the text and
ask it to split on topic changes, returning structured output (the JSON / structured-output technique from Week 2).

```python
# Sketch only — not executed in this notebook
from pydantic import BaseModel

class Chunk(BaseModel):
    title: str      # short topic label, becomes metadata
    text: str       # the exact original text of this chunk

class ChunkList(BaseModel):
    chunks: list[Chunk]

resp = client.beta.chat.completions.parse(   # or client.chat.completions.parse on newer SDKs
    model=CHAT_MODEL,
    response_format=ChunkList,
    messages=[
        {"role": "system", "content": "Split the document into self-contained chunks, one topic each, "
                                      "about 100-300 tokens. Copy text verbatim; do not summarise or rewrite."},
        {"role": "user", "content": raw_text},
    ],
)
llm_chunks = resp.choices[0].message.parsed.chunks
```

A related trick (**contextual chunking**, sometimes called "contextual retrieval"): keep your normal chunks, but for each one ask
the LLM for a 1–2 sentence description of where the chunk fits in the whole document, and prepend it before embedding.
That's the same idea as our `[section title]` prefix, but the LLM writes it.

| | Rule-based (1a / 1b) | LLM-assisted (1c) |
|---|---|---|
| Cost | Free | One LLM call per document (or per chunk for contextual) |
| Speed | Instant | Seconds per document |
| Deterministic | Yes | No — can vary between runs |
| Risk | Bad boundaries on messy text | May paraphrase or drop text — verify output matches the source |
| Long docs | Any size | Must fit in the context window, or be pre-split into windows first |

**Why we don't use it here:** our policy already has clean numbered sections, so 1b gives topic-aligned chunks for free.
Try LLM chunking when the structure isn't there.

---
## Step 2 — Embedding

An **embedding model** turns text into a vector (a list of floats) such that **texts with similar meaning
end up close together** in that vector space — even when they share no words.

`text-embedding-3-small` → 1536 numbers per text.

In [7]:
resp = client.embeddings.create(model=EMBED_MODEL, input="How many days of leave do I get?")
vec = resp.data[0].embedding
print("dimensions:", len(vec))
print("first 8 values:", [round(v, 4) for v in vec[:8]])
print("tokens used:", resp.usage.total_tokens)

dimensions: 1536
first 8 values: [-0.0334, 0.0103, 0.053, 0.0346, -0.0097, 0.0172, -0.0026, -0.0104]
tokens used: 9


### Why this works: cosine similarity
Similarity between two vectors = cosine of the angle between them (1.0 = same direction/meaning, ~0 = unrelated).
Watch how "laptop got stolen" is close to "device theft" even with zero overlapping words:

In [8]:
import math

def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

sentences = [
    "My laptop got stolen from my car",
    "Reporting theft of a company device",
    "Can I install personal software on my work computer?",
    "What is the best pizza topping?",
]
embs = [d.embedding for d in client.embeddings.create(model=EMBED_MODEL, input=sentences).data]

query = sentences[0]
for s, e in zip(sentences[1:], embs[1:]):
    print(f"{cosine(embs[0], e):.3f}  '{query}'  vs  '{s}'")

0.479  'My laptop got stolen from my car'  vs  'Reporting theft of a company device'
0.259  'My laptop got stolen from my car'  vs  'Can I install personal software on my work computer?'
0.020  'My laptop got stolen from my car'  vs  'What is the best pizza topping?'


### Embed all chunks
The API accepts a **list** of inputs, so we embed in batches (one request instead of one per chunk).

In [9]:
def embed_texts(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    vectors = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i : i + batch_size])
        vectors.extend(d.embedding for d in resp.data)
    return vectors

chunk_texts = [c["text"] for c in all_chunks]
chunk_vectors = embed_texts(chunk_texts)

total_tokens = sum(n_tokens(t) for t in chunk_texts)
print(f"embedded {len(chunk_vectors)} chunks x {len(chunk_vectors[0])} dims")
print(f"~{total_tokens:,} tokens -> cost ≈ ${total_tokens / 1_000_000 * 0.02:.5f}")

embedded 57 chunks x 1536 dims
~5,752 tokens -> cost ≈ $0.00012


---
## Step 3 — Store embeddings in a vector database (Chroma)

We could keep vectors in a Python list and loop over them (fine for 50 chunks), but real systems need:
persistence, fast nearest-neighbour search (HNSW index), metadata filtering, and updates.

**Chroma** is an open-source vector DB that can run **embedded and file-based** — no server, just a folder
(like SQLite for vectors). `PersistentClient(path=...)` writes everything to `./chroma_db/`.

Each record in a collection = `id` + `embedding` + `document` (original text) + `metadata`.

In [ ]:
chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

# start fresh each time the notebook runs
if "hr_policies" in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection("hr_policies")

collection = chroma.create_collection(
    name="hr_policies",
    metadata={"hnsw:space": "cosine"},   # use cosine distance (distance = 1 - cosine similarity)
)

for c in all_chunks:
    print(f"Processing chunk with ID: {c['id']}")
    # add each chunk individually to the collection
    collection.add(
        ids=[c["id"]],
        embeddings=[chunk_vectors[all_chunks.index(c)]],  # we pass our own OpenAI vectors
        documents=[chunk_texts[all_chunks.index(c)]],
        metadatas=[c["metadata"]],
    )
collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=chunk_vectors,             # we pass our own OpenAI vectors
    documents=chunk_texts,
    metadatas=[c["metadata"] for c in all_chunks],
)
print("records in collection:", collection.count())

records in collection: 57


It's really just files on disk:

In [11]:
for p in sorted(CHROMA_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(CHROMA_DIR)}  ({p.stat().st_size:,} bytes)")

96ac1fcf-3bb8-4ae2-ac07-d74c7c9235cf/data_level0.bin  (628,400 bytes)
96ac1fcf-3bb8-4ae2-ac07-d74c7c9235cf/header.bin  (100 bytes)
96ac1fcf-3bb8-4ae2-ac07-d74c7c9235cf/length.bin  (400 bytes)
96ac1fcf-3bb8-4ae2-ac07-d74c7c9235cf/link_lists.bin  (0 bytes)
chroma.sqlite3  (999,424 bytes)


Because it's persisted, a completely new client (e.g. your FastAPI app on startup) can reopen it
**without re-chunking or re-embedding**:

In [12]:
reopened = chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection("hr_policies")
print("reopened collection has", reopened.count(), "records")
print(reopened.peek(1)["documents"][0][:200])

reopened collection has 57 records
[PREAMBLE]
IT ASSETS POLICY

Document Owner: IT Department (in coordination with Human Resources)
Version: 1.0
Effective Date: 1 October 2026
Review Cycle: Annually
Applies To: All employees, interns,


---
## Step 4 — A user asks a random question → embed it

**Important:** the question must be embedded with the **same model** used for the chunks,
otherwise the vectors live in different spaces and distances are meaningless.

In [13]:
question = "I spilled coffee on my laptop, what should I do and will I have to pay for it?"

q_vec = client.embeddings.create(model=EMBED_MODEL, input=question).data[0].embedding
print("question vector dims:", len(q_vec))

question vector dims: 1536


## Step 5 — Top-K similarity search
Ask Chroma for the **K nearest chunks** to the question vector. `distance` is cosine distance → **lower = more similar**.

In [14]:
TOP_K = 4
results = collection.query(
    query_embeddings=[q_vec],
    n_results=TOP_K,
    include=["documents", "metadatas", "distances"],
)

# results are lists-of-lists (one inner list per query vector)
for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"distance={dist:.3f}  similarity={1 - dist:.3f}  section={meta['section']}")
    print("   ", doc[:160].replace("\n", " "), "...\n")

distance=0.587  similarity=0.413  section=12. LOSS, THEFT AND DAMAGE
    [12. LOSS, THEFT AND DAMAGE] 12.2 Cost Recovery Each case is reviewed by IT and HR. Cost recovery is decided as follows:  Situation                              ...

distance=0.658  similarity=0.342  section=20. FREQUENTLY ASKED QUESTIONS
    [20. FREQUENTLY ASKED QUESTIONS] Q: When will I get my laptop as a new joiner? A: On your first working day, after you sign the Asset Acknowledgement Form,    p ...

distance=0.677  similarity=0.323  section=12. LOSS, THEFT AND DAMAGE
    [12. LOSS, THEFT AND DAMAGE] 12.1 Reporting If a Company asset is lost or stolen, the employee must: 1. Inform the IT Helpdesk within 24 hours (immediately, if  ...

distance=0.686  similarity=0.314  section=12. LOSS, THEFT AND DAMAGE
    [12. LOSS, THEFT AND DAMAGE] 12.3 Replacement A replacement device is normally provided within 3 working days of reporting. A loaner device may be provided in t ...



The question never said "damage", "repair" or "cost recovery", yet the retrieved chunks come from
**LOSS, THEFT AND DAMAGE** (cost recovery, reporting, replacement) — that's semantic search.

Bonus — metadata filtering (useful once you have many policy files):
```python
collection.query(query_embeddings=[q_vec], n_results=4, where={"source": "it-assets-policy.txt"})
```

## Step 6 — Augment the prompt and call the LLM

Now the "AG" in RAG: put the retrieved chunks into the prompt as **context**, and instruct the model to answer
**only from that context** and cite sections. If the answer isn't in the context, it should say so instead of
making something up.

In [15]:
def build_context(results) -> str:
    blocks = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        blocks.append(f"<source file='{meta['source']}' section='{meta['section']}'>\n{doc}\n</source>")
    return "\n\n".join(blocks)

SYSTEM_PROMPT = '''You are the HR assistant for Rebind Rise.
Answer the employee's question using ONLY the policy excerpts inside <context>.
- Cite the section(s) you used, e.g. (Section 12. LOSS, THEFT AND DAMAGE).
- If the excerpts don't contain the answer, say you don't know and suggest contacting HR/IT. Do not guess.
- Be concise and practical.'''

context = build_context(results)
user_message = f"<context>\n{context}\n</context>\n\nQuestion: {question}"

print(f"prompt context: {n_tokens(context)} tokens  (vs {n_tokens(documents['it-assets-policy.txt']):,} for the whole doc)\n")

completion = client.chat.completions.create(
    model=CHAT_MODEL,
    temperature=0,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ],
)
print(completion.choices[0].message.content)

prompt context: 819 tokens  (vs 5,456 for the whole doc)



You should report the accidental damage to IT as soon as possible. Since this is your first incident of accidental damage within 12 months, you will not have to pay for the repair (Section 12.2).


---
## Putting it all together: `ask()`
Steps 4 → 5 → 6 in one function. This is essentially what would replace "paste the whole policy" in
`hr-assistant/llm/llm_client.py`.

In [16]:
def ask(question: str, k: int = 4, show_sources: bool = True) -> str:
    # 4. embed the question
    q_vec = client.embeddings.create(model=EMBED_MODEL, input=question).data[0].embedding
    # 5. top-K search
    res = collection.query(query_embeddings=[q_vec], n_results=k,
                           include=["documents", "metadatas", "distances"])
    # 6. augment + generate
    completion = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"<context>\n{build_context(res)}\n</context>\n\nQuestion: {question}"},
        ],
    )
    if show_sources:
        print("retrieved:", [f"{m['section']} ({1 - d:.2f})" for m, d in zip(res["metadatas"][0], res["distances"][0])])
    return completion.choices[0].message.content

In [17]:
print(ask("Can I use my personal iPhone to check work email?"))

retrieved: ['9. BRING YOUR OWN DEVICE (BYOD) (0.53)', '9. BRING YOUR OWN DEVICE (BYOD) (0.49)', '9. BRING YOUR OWN DEVICE (BYOD) (0.48)', '9. BRING YOUR OWN DEVICE (BYOD) (0.48)']
Yes, you may access limited Company services, including email, from your personal mobile phone, such as an iPhone, as long as you follow the requirements outlined in the policy (Section 9.1 Permitted Use and Section 9.2 Requirements).


In [18]:
print(ask("When do I become eligible for a new laptop?"))

retrieved: ['20. FREQUENTLY ASKED QUESTIONS (0.57)', '6. LAPTOPS AND DESKTOPS (0.52)', '6. LAPTOPS AND DESKTOPS (0.52)', '5. ASSET ENTITLEMENT (0.44)']
You may be eligible for a new laptop if your current device is failing, cannot run required software, or is no longer supported with security updates. Additionally, laptops are replaced every 3 to 4 years depending on the type (Section 6.5). For an early replacement review, you should raise a ticket with IT.


In [19]:
print(ask("I'm leaving the company next week. What do I need to do with my equipment?"))

retrieved: ['16. RETURN OF ASSETS ON EXIT (0.53)', '15. TRANSFERS AND ROLE CHANGES (0.51)', '16. RETURN OF ASSETS ON EXIT (0.50)', '4. ROLES AND RESPONSIBILITIES (0.47)']
You must return all Company assets on or before your last working day. If you are a remote employee, you will receive a prepaid courier kit and must ship the assets within 3 working days of your last working day. Additionally, do not reset, wipe, or reformat any devices yourself; IT will perform a secure wipe after any required data has been backed up (Section 16. RETURN OF ASSETS ON EXIT).


### Out-of-scope question
`leave-policy.txt` is empty, so nothing about leave is indexed. Top-K **always** returns K chunks —
even if none are relevant (look at the low similarity scores). The system prompt is what stops the model
from hallucinating an answer.

In [20]:
print(ask("How many days of maternity leave do I get?"))

retrieved: ['7. MOBILE PHONES AND TABLETS (0.21)', '16. RETURN OF ASSETS ON EXIT (0.21)', '12. LOSS, THEFT AND DAMAGE (0.20)', '4. ROLES AND RESPONSIBILITIES (0.20)']
I don't know. Please contact HR for information regarding maternity leave.


---
## Recap

| Step | What we did | Key choice |
|---|---|---|
| 1. Chunk | Split policy by section → sub-section → paragraph, prefixed section title | chunk size, overlap, structure-aware vs fixed |
| 2. Embed | `text-embedding-3-small`, batched | same model for chunks **and** queries |
| 3. Store | Chroma `PersistentClient` → `./chroma_db/` folder, cosine space | metadata for citations/filters |
| 4. Embed query | one API call per question | — |
| 5. Retrieve | `collection.query(n_results=K)` | K (too low → miss info, too high → noise/cost) |
| 6. Generate | context + strict "answer only from context" prompt | citations, "I don't know" behaviour |

## Try it yourself
1. Change `max_tokens` in `chunk_document` to 100 and 800 — how do retrieved sections and answers change?
2. Use `fixed_size_chunks` instead and compare answers for the same questions.
3. Add a **similarity threshold**: drop chunks with similarity < 0.3 and, if nothing is left, answer "I don't know" without calling the LLM.
4. Write some content into `leave-policy.txt`, re-run the indexing cells, and ask the maternity leave question again.
5. Wire `ask()` into `hr-assistant` so it no longer sends the whole IT policy on every request.